Step 1: Preprocessing
- Tokenize the documents
- Remove stop words
- (Optional) Stemming/Lemmatization

Boolean Retrieval Model

In [1]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def preprocess(text):
    # Tokenize and remove non-alphanumeric
    words = re.sub(r'[^a-zA-Z0-9\s]', '', text.lower()).split()
    # Remove stop words
    return [w for w in words if w not in stop_words]

# Sample Documents
documents = {
    'Doc1': 'The quick brown fox jumps over the lazy dog',
    'Doc2': 'Never jump over the lazy dog',
    'Doc3': 'The fox is quick and brown'
}

processed_docs = {doc_id: preprocess(content) for doc_id, content in documents.items()}
print("Processed Documents:", processed_docs)

Processed Documents: {'Doc1': ['quick', 'brown', 'fox', 'jumps', 'lazy', 'dog'], 'Doc2': ['never', 'jump', 'lazy', 'dog'], 'Doc3': ['fox', 'quick', 'brown']}


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


Step 2: Create a Term-Document Matrix
- List all unique terms across all documents
- Create a binary matrix (1 if term is present, 0 otherwise)

In [2]:
# Create Term-Document Matrix
all_terms = sorted(list(set([term for doc in processed_docs.values() for term in doc])))

matrix = {}
for term in all_terms:
    matrix[term] = [1 if term in processed_docs[doc_id] else 0 for doc_id in documents]

df_matrix = pd.DataFrame(matrix, index=documents.keys())
print("Term-Document Matrix:")
display(df_matrix)

Term-Document Matrix:


,brown,dog,fox,jump,jumps,lazy,never,quick
Doc1,1,1,1,0,1,1,0,1
Doc2,0,1,0,1,0,1,1,0
Doc3,1,0,1,0,0,0,0,1


### Step 3: Represent the Query in DNF
Disjunctive Normal Form (DNF) is represented as a union of intersections (e.g., (A AND B) OR (C AND D)).

In [3]:
def evaluate_dnf(dnf_query):
    """
    Evaluates a query in DNF format.
    Input format: [['term1', 'term2'], ['term3']]
    Represents: (term1 AND term2) OR (term3)
    """
    dnf_results = []
    for conjunction in dnf_query:
        # Step 3.1: Handle the AND part of the conjunction
        res = boolean_query(conjunction, 'AND')
        dnf_results.append(res)

    # Step 3.2: Handle the OR part across conjunctions
    final_result = dnf_results[0]
    for res in dnf_results[1:]:
        final_result = final_result | res
    return final_result

In [4]:
def boolean_query(query_terms, operator='AND'):
    """Simplistic Boolean evaluator for DNF components."""
    results = []
    for term in query_terms:
        if term in df_matrix.columns:
            results.append(df_matrix[term])
        else:
            results.append(pd.Series([0]*len(df_matrix), index=df_matrix.index))

    if not results: return None

    final_res = results[0]
    for res in results[1:]:
        if operator == 'AND':
            final_res = final_res & res
        else:
            final_res = final_res | res
    return final_res

# Example: (quick AND brown) OR (jump)
clause1 = boolean_query(['quick', 'brown'], 'AND')
clause2 = boolean_query(['jump'], 'OR')
result = clause1 | clause2

print("Query Result ((quick AND brown) OR jump):")
print(result[result == 1].index.tolist())

Query Result ((quick AND brown) OR jump):
['Doc1', 'Doc2', 'Doc3']


### Step 4: Apply Similarity Formula and Display Results

In [5]:
# Example Query in DNF: (quick AND brown) OR (jump)
query_dnf = [['quick', 'brown'], ['jump']]

final_matches = evaluate_dnf(query_dnf)

print(f"Documents matching the DNF query {query_dnf}:")
print(final_matches[final_matches == 1].index.tolist())

Documents matching the DNF query [['quick', 'brown'], ['jump']]:
['Doc1', 'Doc2', 'Doc3']
